In [ ]:
import fastf1
import pandas as pd
import sklearn as sk

In [85]:
CACHE_DIR = "/Users/lucas/Documents/Code/fastf1_cache"
fastf1.Cache.enable_cache(CACHE_DIR)

In [ ]:
def load_gp(year, rnd):
    r = fastf1.get_session(year, rnd, "R")
    r.load(laps=False, telemetry=False, weather=False, messages=False)
    race = r.results[["Abbreviation", "TeamName", "GridPosition",
                      "Position", "Status", "Points", ]].copy()
    race = race.rename(columns={"Position": "FinishPosition"})
    qs = fastf1.get_session(year, rnd, "Q")
    qs.load(laps=False, telemetry=False, weather=False, messages=False)
    q = qs.results[["Abbreviation", "Position", "Q1", "Q2", "Q3"]].copy()
    q = q.rename(columns={"Position": "QualiPosition"})
    for col in ["Q1", "Q2", "Q3"]:
        q[col] = q[col].dt.total_seconds()
    df = race.merge(q, on="Abbreviation", how="left")
    df["Year"] = year
    df["Round"] = rnd
    df["EventName"] = r.event["EventName"]
    df["Country"] = r.event["Country"]
    df["EventDate"] = r.event["EventDate"]
    return df



all_gps = []
for year in range(2018, 2020):
    sched = fastf1.get_event_schedule(year, include_testing=False)
    for rnd in sched["RoundNumber"]:
        df = load_gp(year, int(rnd))
        if df is not None:
            all_gps.append(df)
            print(f"✅ {year} - {rnd}")

data = pd.concat(all_gps, ignore_index=True)
print(data.shape)

In [ ]:
X = data[["GridPosition", "Q1", "Q2", "Q3"]]
y = data["FinishPosition"]


,GridPosition,Q1,Q2,Q3
0,3.0,83.348,81.944,81.838
1,1.0,82.824,82.051,81.164
2,2.0,83.096,82.507,81.828
3,8.0,83.494,82.897,82.152
4,10.0,83.597,83.692,NaN


In [116]:
preparation = sk.compose.ColumnTransformer([
    ("nombres", sk.impute.SimpleImputer(strategy="median"), numeric),
    ("categories", sk.pipeline.Pipeline([
        ("remplacer_manquants", sk.impute.SimpleImputer(strategy="most_frequent")),
        ("encoder", sk.preprocessing.OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
])

modele = sk.pipeline.Pipeline([
    ("preparation", preparation),
    ("foret", sk.ensemble.GradientBoostingClassifier()),
])

X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modele.fit(X_train, y_train)
predictions = modele.predict(X_test)
print(sk.metrics.classification_report(y_test, predictions, zero_division=0))

ValueError: Some column names are not columns of the dataframe: {'Year', 'QualiPosition', 'last_3_races_avg_points'}